Agentic RAG PDF Chatbot

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.tools import tool
from langchain.agents import create_agent


doc_load = PyPDFLoader("../data/Sample Blood Test Report.pdf")
pdf_load = doc_load.load()
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 100)
split_doc = splitter.split_documents(pdf_load)
embedding = OpenAIEmbeddings(model = "text-embedding-3-small")
vector_store = InMemoryVectorStore.from_documents(documents = split_doc,
                                   embedding = embedding)
query = "What is the name of the patient and doctor?"
@tool
def get_context(query:str):
    """
        This tool can help you to retrieve the relevant data of the PDF Documents, and these pdf
        documents have details about medical reports.
    """
    similarity = vector_store.similarity_search(query)
    print(f"get_context called {query}")
    context = ""
    for doc in similarity:
        context = doc.page_content

        return context

#get_context(query=query)
system_prompt = "You are a helpful assistant that answers questions using retrieved context. ALWAYS use the 'get_context' tool for questions requiring external knowledge."
llm = ChatOpenAI(model = "gpt-5")
agent = create_agent(model = llm,
                     tools = [get_context],
                     system_prompt= system_prompt)
result = agent.invoke({"messages":{"role":"user", "content":query}})
print(result["messages"][-1].content)



get_context called patient name and doctor name
- Patient: Eleanor Vance
- Doctor: Dr. Robert H. Chen, MD
